# 03 — Gold Aggregation: Predictive Maintenance Analytics

This notebook reads cleaned data from the **Silver** lakehouse and produces
business-ready aggregated tables for predictive maintenance analytics in **Gold**.

**Output Tables:**
- `agg_vehicle_health_score` — per-vehicle health score based on sensors + DTCs
- `agg_fleet_daily_summary` — daily fleet-level KPIs
- `agg_maintenance_cost_analysis` — cost breakdown by type, category, fleet
- `agg_dtc_frequency` — DTC code frequency and trend analysis

**Medallion Layer:** Gold (aggregated, business-ready, V-Order optimized)

In [ ]:
# Gold layer Spark config: read-optimized with V-Order
spark.conf.set("spark.sql.parquet.vorder.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.sql.adaptive.enabled", "true")

In [ ]:
from pyspark.sql.functions import (
    col, avg, count, sum as _sum, min as _min, max as _max,
    when, lit, current_timestamp, to_date, datediff,
    round as _round, countDistinct, desc, percent_rank,
    concat_ws, collect_set
)
from pyspark.sql.window import Window

In [ ]:
# Read Silver tables from the Silver lakehouse
# NOTE: These IDs are replaced at deploy time by scripts/deploy.ps1
SILVER_BASE = "abfss://{{WORKSPACE_ID}}@onelake.dfs.fabric.microsoft.com/{{SILVER_LAKEHOUSE_ID}}"

df_vehicles = spark.read.format("delta").load(f"{SILVER_BASE}/Tables/dim_vehicles")
df_telemetry = spark.read.format("delta").load(f"{SILVER_BASE}/Tables/fact_telemetry")
df_dtcs = spark.read.format("delta").load(f"{SILVER_BASE}/Tables/fact_dtc_events")
df_maintenance = spark.read.format("delta").load(f"{SILVER_BASE}/Tables/fact_maintenance")

print("Silver tables loaded successfully")

## Vehicle Health Score

Composite score (0–100) per vehicle based on:
- Sensor anomaly frequency (40%)
- DTC severity (30%)
- Maintenance recency (30%)

In [ ]:
# Sensor anomaly metrics per vehicle
df_sensor_metrics = (
    df_telemetry
    .groupBy("vin")
    .agg(
        count("*").alias("total_readings"),
        _sum(when(col("engine_temp_anomaly") == True, 1).otherwise(0)).alias("engine_temp_anomaly_count"),
        _sum(when(col("oil_pressure_anomaly") == True, 1).otherwise(0)).alias("oil_pressure_anomaly_count"),
        _sum(when(col("battery_anomaly") == True, 1).otherwise(0)).alias("battery_anomaly_count"),
        avg("engine_temp_f").alias("avg_engine_temp"),
        avg("oil_pressure_psi").alias("avg_oil_pressure"),
        avg("battery_voltage_v").alias("avg_battery_voltage"),
        avg("rpm").alias("avg_rpm")
    )
    .withColumn("anomaly_rate",
        (col("engine_temp_anomaly_count") + col("oil_pressure_anomaly_count") + col("battery_anomaly_count"))
        / col("total_readings")
    )
)

# DTC severity score per vehicle
df_dtc_score = (
    df_dtcs
    .groupBy("vin")
    .agg(
        count("*").alias("total_dtcs"),
        _sum("severity_rank").alias("total_severity_points"),
        countDistinct("dtc_code").alias("unique_dtc_codes"),
        _sum(when(col("severity") == "Critical", 1).otherwise(0)).alias("critical_dtc_count")
    )
)

# Maintenance recency per vehicle
df_maint_score = (
    df_maintenance
    .groupBy("vin")
    .agg(
        count("*").alias("total_services"),
        _max("service_date").alias("last_service_date"),
        _sum("cost_usd").alias("total_maintenance_cost")
    )
    .withColumn("days_since_last_service",
        datediff(current_timestamp(), col("last_service_date")))
)

# Combine into health score
df_health = (
    df_vehicles.select("vin", "make", "model", "year", "mileage", "region", "fleet_id")
    .join(df_sensor_metrics, "vin", "left")
    .join(df_dtc_score, "vin", "left")
    .join(df_maint_score, "vin", "left")
    .fillna(0)
)

# Compute percentile ranks for scoring components
w_all = Window.orderBy("vin")
df_health = (
    df_health
    # Sensor score: lower anomaly rate = higher score
    .withColumn("sensor_score", _round((1 - col("anomaly_rate")) * 100, 1))
    # DTC score: fewer severity points = higher score
    .withColumn("dtc_score",
        _round(when(col("total_severity_points") == 0, lit(100))
        .otherwise(when(col("total_severity_points") > 50, lit(0))
        .otherwise((1 - col("total_severity_points") / 50) * 100)), 1))
    # Maintenance score: recent service = higher score
    .withColumn("maintenance_score",
        _round(when(col("days_since_last_service") <= 30, lit(100))
        .when(col("days_since_last_service") <= 60, lit(75))
        .when(col("days_since_last_service") <= 90, lit(50))
        .otherwise(lit(25)), 1))
    # Composite health score (weighted)
    .withColumn("health_score",
        _round(col("sensor_score") * 0.4 + col("dtc_score") * 0.3 + col("maintenance_score") * 0.3, 1))
    # Risk level
    .withColumn("risk_level",
        when(col("health_score") >= 80, lit("Low"))
        .when(col("health_score") >= 50, lit("Medium"))
        .otherwise(lit("High")))
    .withColumn("_aggregated_at", current_timestamp())
)

df_health.write.format("delta").mode("overwrite").save("Tables/agg_vehicle_health_score")
print(f"Gold agg_vehicle_health_score: {df_health.count()} rows")
df_health.groupBy("risk_level").count().show()

## Fleet Daily Summary

In [ ]:
df_daily = (
    df_telemetry
    .withColumn("reading_date", to_date("timestamp"))
    .join(df_vehicles.select("vin", "fleet_id", "region"), "vin", "left")
    .groupBy("reading_date", "fleet_id", "region")
    .agg(
        countDistinct("vin").alias("active_vehicles"),
        count("*").alias("total_readings"),
        _round(avg("engine_temp_f"), 1).alias("avg_engine_temp"),
        _round(avg("oil_pressure_psi"), 1).alias("avg_oil_pressure"),
        _round(avg("battery_voltage_v"), 2).alias("avg_battery_voltage"),
        _round(avg("speed_mph"), 1).alias("avg_speed"),
        _sum(when(col("engine_temp_anomaly") == True, 1).otherwise(0)).alias("engine_anomaly_count"),
        _sum(when(col("oil_pressure_anomaly") == True, 1).otherwise(0)).alias("oil_anomaly_count"),
        _sum(when(col("battery_anomaly") == True, 1).otherwise(0)).alias("battery_anomaly_count")
    )
    .withColumn("total_anomalies",
        col("engine_anomaly_count") + col("oil_anomaly_count") + col("battery_anomaly_count"))
    .withColumn("anomaly_rate_pct",
        _round(col("total_anomalies") / col("total_readings") * 100, 2))
    .withColumn("_aggregated_at", current_timestamp())
    .orderBy("reading_date", "fleet_id")
)

df_daily.write.format("delta").mode("overwrite").save("Tables/agg_fleet_daily_summary")
print(f"Gold agg_fleet_daily_summary: {df_daily.count()} rows")

## Maintenance Cost Analysis

In [ ]:
df_cost = (
    df_maintenance
    .join(df_vehicles.select("vin", "make", "model", "year", "fleet_id", "region"), "vin", "left")
    .groupBy("fleet_id", "region", "make", "model", "maintenance_type", "category")
    .agg(
        count("*").alias("service_count"),
        _round(_sum("cost_usd"), 2).alias("total_cost"),
        _round(avg("cost_usd"), 2).alias("avg_cost"),
        _round(_sum("labor_hours"), 1).alias("total_labor_hours"),
        _round(avg("labor_hours"), 1).alias("avg_labor_hours"),
        _sum("parts_replaced").alias("total_parts_replaced"),
        _sum(when(col("warranty_claim") == True, 1).otherwise(0)).alias("warranty_claims"),
        _sum(when(col("warranty_claim") == True, col("cost_usd")).otherwise(0)).alias("warranty_cost")
    )
    .withColumn("warranty_rate_pct",
        _round(col("warranty_claims") / col("service_count") * 100, 1))
    .withColumn("_aggregated_at", current_timestamp())
)

df_cost.write.format("delta").mode("overwrite").save("Tables/agg_maintenance_cost_analysis")
print(f"Gold agg_maintenance_cost_analysis: {df_cost.count()} rows")

## DTC Frequency & Trend Analysis

In [ ]:
df_dtc_freq = (
    df_dtcs
    .withColumn("event_date", to_date("timestamp"))
    .join(df_vehicles.select("vin", "make", "model", "fleet_id", "region"), "vin", "left")
    .groupBy("dtc_code", "dtc_description", "system_category", "severity",
             "event_date", "fleet_id", "region", "make")
    .agg(
        count("*").alias("occurrence_count"),
        countDistinct("vin").alias("affected_vehicles"),
        _sum(when(col("is_active") == True, 1).otherwise(0)).alias("active_count")
    )
    .withColumn("active_rate_pct",
        _round(col("active_count") / col("occurrence_count") * 100, 1))
    .withColumn("_aggregated_at", current_timestamp())
    .orderBy(desc("occurrence_count"))
)

df_dtc_freq.write.format("delta").mode("overwrite").save("Tables/agg_dtc_frequency")
print(f"Gold agg_dtc_frequency: {df_dtc_freq.count()} rows")

## Summary

In [ ]:
print("=== Gold Layer Aggregation Complete ===")
print(f"  agg_vehicle_health_score:      {spark.read.format('delta').load('Tables/agg_vehicle_health_score').count()} rows")
print(f"  agg_fleet_daily_summary:       {spark.read.format('delta').load('Tables/agg_fleet_daily_summary').count()} rows")
print(f"  agg_maintenance_cost_analysis: {spark.read.format('delta').load('Tables/agg_maintenance_cost_analysis').count()} rows")
print(f"  agg_dtc_frequency:             {spark.read.format('delta').load('Tables/agg_dtc_frequency').count()} rows")
print("\nHealth Score Distribution:")
spark.read.format("delta").load("Tables/agg_vehicle_health_score") \
    .groupBy("risk_level").count().show()